In [1]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
RANDOMSEED=1727

os.environ['PYTHONHASHSEED'] = str(RANDOMSEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ":4096:2"

In [2]:
import torch,random
import tensorflow as tf
import numpy as np

tf.config.experimental.enable_op_determinism()
torch.use_deterministic_algorithms(True)

def random_seed(seed=RANDOMSEED, use_cuda=False):
  np.random.seed(seed) # cpu vars
  torch.manual_seed(seed) # cpu vars
  random.seed(seed) # Python
  tf.random.set_seed(seed)
  tf.keras.utils.set_random_seed(seed)
  
  if use_cuda: 
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # gpu vars
    torch.backends.cudnn.deterministic = True  #needed
    torch.backends.cudnn.benchmark = False
    
random_seed(RANDOMSEED)
tf.config.set_visible_devices([], 'GPU')

In [3]:
from pathlib import Path

cred_path = Path('~/.kaggle/access_token').expanduser()
if not cred_path.exists():
  cred_path.parent.mkdir(exist_ok=True)
  cred_path.write_text("KGAT_9f6b15aaf6f7637b8497dfb3c56c079e")
  cred_path.chmod(0o600)

In [4]:
nname = Path('nyc-taxi-trip-duration')

if not iskaggle:
  if not nname.exists():
    import zipfile,kaggle
    kaggle.api.authenticate()
    kaggle.api.competition_download_cli(str(nname))
    zipfile.ZipFile(f'{nname}.zip').extractall(nname)
else:
  # /kaggle/input/competitions/nlp-getting-started/train.csv
  nname = Path(f'/kaggle/input/competitions/{nname}')

# %pip install -q datasets
!dir /o:g /w {nname}
# !ls {nname}

100%|██████████| 85.8M/85.8M [00:11<00:00, 7.86MB/s]



 Volume in drive C is Windows
 Volume Serial Number is 6291-898F

 Directory of c:\Users\longnuub\learning-programming-languages\learning-python\kaggle\nyc-taxi-trip-duration

[..]                    [.]                     test.zip
train.zip               sample_submission.zip   
               3 File(s)     89.910.233 bytes
               2 Dir(s)  136.330.530.816 bytes free


In [23]:
import pandas as pd
for dirpath,_,filenames in os.walk(nname):
    for filename in filenames:
        name,ext = os.path.splitext(filename)
        if ext==".zip":
            print(f"found zip: {filename}")
            zipfile.ZipFile(f"{os.path.join(dirpath,filename)}").extractall(dirpath)

found zip: sample_submission.zip
found zip: test.zip
found zip: train.zip


In [38]:
train=pd.read_csv(nname/"train.csv")
test=pd.read_csv(nname/"test.csv")

In [39]:
# drop the `id` col for training data ONLY, we need the id for test preds later
# train.drop(columns=["id"],inplace=True)
# test
train.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [40]:
# xtrain=train.copy().drop(columns="trip_duration")
ytrain=train["trip_duration"]
xtest=test.copy()

In [41]:
train.isna().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [42]:
pickd=pd.to_datetime(train["pickup_datetime"])
dropd=pd.to_datetime(train["dropoff_datetime"])
trip_d=pd.to_numeric(train["trip_duration"])
tripd=(dropd-pickd).dt.total_seconds()
delta_tripd_vs_trip_d=tripd-trip_d
delta_tripd_vs_trip_d.describe()

count    1458644.0
mean           0.0
std            0.0
min            0.0
25%            0.0
50%            0.0
75%            0.0
max            0.0
dtype: float64

In [43]:
xtrain=train.drop(["pickup_datetime","dropoff_datetime"],axis=1)